In [1]:
import sys
import openmc

from openmc.data.endf import Evaluation, get_head_record, get_tab1_record

from pathlib import Path
from io import StringIO

print(f"Python version: {sys.version.split()[0]}")
print(f"OpenMC version: {openmc.__version__}")

Python version: 3.12.13
OpenMC version: 0.15.3


In [2]:
DATA_ROOT = Path("/home/mathew-oaks/Desktop/nuclear-data-validation/data")

JEFF40_DIR = DATA_ROOT / "JEFF-4.0"
EAF_DIR    = DATA_ROOT / "EAF-2010"


def find_single_data_file(directory):
    """Return the single non-hidden file stored in a library directory."""
    if not directory.exists():
        raise FileNotFoundError(
            f"Missing directory: {directory}\n"
            "Create it and place the corresponding Mg-24 ENDF file inside."
        )

    files = [
        path for path in directory.iterdir()
        if path.is_file() and not path.name.startswith(".")
    ]

    if len(files) != 1:
        raise RuntimeError(
            f"{directory} must contain exactly one evaluated Mg-24 file; "
            f"found {len(files)}."
        )

    return files[0]


JEFF40 = find_single_data_file(JEFF40_DIR)
EAF    = find_single_data_file(EAF_DIR)

print("Data files:")
print(f"  JEFF-4.0: {JEFF40}")
print(f"  EAF-2010: {EAF}")

Data files:
  JEFF-4.0: /home/mathew-oaks/Desktop/nuclear-data-validation/data/JEFF-4.0/n_12-Mg-024g.jeff
  EAF-2010: /home/mathew-oaks/Desktop/nuclear-data-validation/data/EAF-2010/Mg24.endf


In [3]:
jeff40_eval = Evaluation(JEFF40)
eaf_eval    = Evaluation(EAF)

evaluations = {
    "JEFF-4.0 Mg-24": jeff40_eval,
    "EAF-2010 Mg-24": eaf_eval,
}

print("All three Mg-24 evaluations loaded successfully.")

All three Mg-24 evaluations loaded successfully.


In [4]:
def check_mt103_sections(ev, label):
    print(f"\n{'='*60}")
    print(label)
    print(f"{'='*60}")

    for mf in (9, 10):
        key = (mf, 103)

        if key in ev.section:
            print(f"FOUND: MF={mf}, MT=103")
        else:
            print(f"NOT FOUND: MF={mf}, MT=103")


for label, ev in evaluations.items():
    check_mt103_sections(ev, label)


JEFF-4.0 Mg-24
NOT FOUND: MF=9, MT=103
NOT FOUND: MF=10, MT=103

EAF-2010 Mg-24
NOT FOUND: MF=9, MT=103
FOUND: MF=10, MT=103


In [5]:
def inspect_mf9_mt103(ev, label):
    key = (9, 103)

    print(f"\n{'='*70}")
    print(f"{label}: MF=9, MT=103")
    print(f"{'='*70}")

    if key not in ev.section:
        print("MF=9, MT=103 is not present.")
        return

    file_obj = StringIO(ev.section[key])

    # MF=9 HEAD:
    # ZA, AWR, LIS, 0, NS, 0
    head = get_head_record(file_obj)

    ZA  = head[0]
    AWR = head[1]
    LIS = head[2]
    NS  = head[4]

    print(f"ZA  = {ZA}")
    print(f"AWR = {AWR}")
    print(f"LIS = {LIS}")
    print(f"Number of final states NS = {NS}")

    for i in range(NS):
        params, branching = get_tab1_record(file_obj)

        QM   = params[0]
        QI   = params[1]
        IZAP = int(params[2])
        LFS  = int(params[3])

        print(f"\nFinal state {i + 1}")
        print(f"  IZAP = {IZAP}")
        print(f"  LFS  = {LFS}")
        print(f"  QM   = {QM:.6e} eV")
        print(f"  QI   = {QI:.6e} eV")

        if LFS == 0:
            print("  State = ground state")
        else:
            print(f"  State = excited/metastable state LFS={LFS}")

        print("  Incident energy [eV]     Branching fraction")
        for E, y in zip(branching.x, branching.y):
            print(f"  {E:16.8e}     {y:16.8e}")


for label, ev in evaluations.items():
    inspect_mf9_mt103(ev, label)


JEFF-4.0 Mg-24: MF=9, MT=103
MF=9, MT=103 is not present.

EAF-2010 Mg-24: MF=9, MT=103
MF=9, MT=103 is not present.


In [6]:
def inspect_mf10_mt103(ev, label):
    key = (10, 103)

    print(f"\n{'='*70}")
    print(f"{label}: MF=10, MT=103")
    print(f"{'='*70}")

    if key not in ev.section:
        print("MF=10, MT=103 is not present.")
        return

    file_obj = StringIO(ev.section[key])

    # MF=10 HEAD:
    # ZA, AWR, LIS, 0, NS, 0
    head = get_head_record(file_obj)

    ZA  = head[0]
    AWR = head[1]
    LIS = head[2]
    NS  = head[4]

    print(f"ZA  = {ZA}")
    print(f"AWR = {AWR}")
    print(f"LIS = {LIS}")
    print(f"Number of final states NS = {NS}")

    for i in range(NS):
        params, cross_section = get_tab1_record(file_obj)

        QM   = params[0]
        QI   = params[1]
        IZAP = int(params[2])
        LFS  = int(params[3])

        print(f"\nFinal state {i + 1}")
        print(f"  IZAP = {IZAP}")
        print(f"  LFS  = {LFS}")
        print(f"  QM   = {QM:.6e} eV")
        print(f"  QI   = {QI:.6e} eV")

        if LFS == 0:
            print("  State = ground state")
        else:
            print(f"  State = excited/metastable state LFS={LFS}")

        print("  Incident energy [eV]     Production cross section [barns]")
        for E, xs in zip(cross_section.x, cross_section.y):
            print(f"  {E:16.8e}     {xs:16.8e}")


for label, ev in evaluations.items():
    inspect_mf10_mt103(ev, label)


JEFF-4.0 Mg-24: MF=10, MT=103
MF=10, MT=103 is not present.

EAF-2010 Mg-24: MF=10, MT=103
ZA  = 12024
AWR = 23.779
LIS = 0
Number of final states NS = 2

Final state 1
  IZAP = 11024
  LFS  = 0
  QM   = -4.733430e+06 eV
  QI   = -4.733430e+06 eV
  State = ground state
  Incident energy [eV]     Production cross section [barns]
    4.93248900e+06       0.00000000e+00
    4.94117000e+06       2.07000000e-05
    5.40000000e+06       2.76000000e-05
    5.75000000e+06       1.24200000e-04
    6.00000000e+06       1.53180000e-03
    6.20000000e+06       3.16020000e-03
    6.40000000e+06       5.89260000e-03
    6.60000000e+06       2.74206000e-02
    6.80000000e+06       3.04497000e-02
    7.00000000e+06       2.49573000e-02
    7.20500000e+06       3.75567000e-02
    7.41000000e+06       3.25680000e-02
    7.61000000e+06       5.56554000e-02
    7.86000000e+06       6.44598000e-02
    8.15500000e+06       8.44353000e-02
    8.45000000e+06       8.20272000e-02
    8.80000000e+06       8.84

In [7]:
NA24_IZAP = 11024


def get_mt103_states(ev, mf):
    """Return the product-state records in MF=9 or MF=10, MT=103."""
    key = (mf, 103)

    if key not in ev.section:
        return []

    file_obj = StringIO(ev.section[key])
    head = get_head_record(file_obj)
    NS = head[4]

    states = []

    for _ in range(NS):
        params, data = get_tab1_record(file_obj)

        states.append({
            "IZAP": int(params[2]),
            "LFS": int(params[3]),
            "QM": params[0],
            "QI": params[1],
        })

    return states


print("\n" + "="*78)
print("FINAL MT=103 VERIFICATION SUMMARY")
print("="*78)
print("Target state check: Na-24 with IZAP=11024 and LFS>0")
print()

for label, ev in evaluations.items():
    mf9_present = (9, 103) in ev.section
    mf10_present = (10, 103) in ev.section

    mf9_states = get_mt103_states(ev, 9)
    mf10_states = get_mt103_states(ev, 10)

    na24_excited_mf9 = any(
        state["IZAP"] == NA24_IZAP and state["LFS"] > 0
        for state in mf9_states
    )

    na24_excited_mf10 = any(
        state["IZAP"] == NA24_IZAP and state["LFS"] > 0
        for state in mf10_states
    )

    explicit_state_resolved_entry = na24_excited_mf9 or na24_excited_mf10

    print(label)
    print(f"  MF=9,  MT=103 present: {'YES' if mf9_present else 'NO'}")
    print(f"  MF=10, MT=103 present: {'YES' if mf10_present else 'NO'}")
    print(
        "  Na-24 LFS>0 entry in MF=9/MT=103: "
        f"{'YES' if na24_excited_mf9 else 'NO'}"
    )
    print(
        "  Na-24 LFS>0 entry in MF=10/MT=103: "
        f"{'YES' if na24_excited_mf10 else 'NO'}"
    )
    print(
        "  Explicit state-resolved Na-24 excited/metastable production entry: "
        f"{'YES' if explicit_state_resolved_entry else 'NO'}"
    )
    print()

print("="*78)


FINAL MT=103 VERIFICATION SUMMARY
Target state check: Na-24 with IZAP=11024 and LFS>0

JEFF-4.0 Mg-24
  MF=9,  MT=103 present: NO
  MF=10, MT=103 present: NO
  Na-24 LFS>0 entry in MF=9/MT=103: NO
  Na-24 LFS>0 entry in MF=10/MT=103: NO
  Explicit state-resolved Na-24 excited/metastable production entry: NO

EAF-2010 Mg-24
  MF=9,  MT=103 present: NO
  MF=10, MT=103 present: YES
  Na-24 LFS>0 entry in MF=9/MT=103: NO
  Na-24 LFS>0 entry in MF=10/MT=103: YES
  Explicit state-resolved Na-24 excited/metastable production entry: YES

